In [1]:
from langchain_core.tools import tool
from langchain_aws import ChatBedrockConverse
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver


In [2]:
@tool
def check_course_availability(course_name: str):
    """Checks if seats are available for a specific training course."""
    
    # Mock data for a training institute
    courses = {
        "Data Science": "5 seats left (Batch starts Monday)",
        "Web Development": "Waitlist only",
        "AI Ethics": "12 seats available"
    }
    return courses.get(course_name, "Course not found. Please ask about Data Science or Web Dev.")

@tool
def calculate_installment_plan(total_fee: float, months: int):
    """Calculates the monthly payment for a student fee plan."""
    if months <= 0:
        return "Duration must be at least 1 month."
    monthly = total_fee / months
    return f"The installment plan is {months} payments of ${monthly:.2f} per month."
    
tools = [check_course_availability, calculate_installment_plan]


In [3]:
model = ChatBedrockConverse(
	model_id="cohere.command-r-plus-v1:0",
	region_name="us-east-1",
	temperature=0.3
	)


In [4]:
summary_model = ChatBedrockConverse(
	model_id="amazon.nova-lite-v1:0",
	region_name="us-east-1"
	)


In [5]:
SYSTEM_PROMPT = """You are the professional receptionist for 'FutureTech Institute'.
Your goal is to help prospective students with course info and fee queries.
Be polite, helpful, and concise."""
agent = create_agent(
	model=model,
	tools=tools,
	system_prompt=SYSTEM_PROMPT,
	checkpointer=InMemorySaver(),
	middleware=[
		SummarizationMiddleware(
		model=summary_model,
		trigger=("tokens", 400), # Summarize when context hits 400 tokens
		keep=("messages", 6), # Always keep the last 6 messages raw
		),
		],
	)


In [6]:
from rich import print
config = {"configurable": {"thread_id": "student_001"}}
while True:
	query=input("User_Query:")
	if query=='exit':
		break
	response = agent.invoke(
		{"messages": [{"role": "user", "content":query }]},
		config=config
		)
		
	print(response["messages"][-1].content)


User_Query: Hi


Hello, how can I help?

User_Query: I want to know seats available for AI cousrse


Apologies, I can't find an AI course. Would you like to ask about Data Science or Web Dev instead?

User_Query: I want to know seats available for AI Ethics


There are 12 seats available for the AI Ethics course.

User_Query: for data Science course


There are 5 seats left for the Data Science course. The batch starts on Monday.

User_Query: exit


In [7]:
print(response)

{
    'messages': [
        HumanMessage(
            content='Hi',
            additional_kwargs={},
            response_metadata={},
            id='e0f08b82-7bde-4bbf-a825-bb5969c575ca'
        ),
        AIMessage(
            content='Hello, how can I help?',
            additional_kwargs={},
            response_metadata={
                'ResponseMetadata': {
                    'RequestId': '75ed703e-5f94-40b0-9f24-339f4943f31e',
                    'HTTPStatusCode': 200,
                    'HTTPHeaders': {
                        'date': 'Sat, 02 May 2026 17:25:20 GMT',
                        'content-type': 'application/json',
                        'content-length': '225',
                        'connection': 'keep-alive',
                        'x-amzn-requestid': '75ed703e-5f94-40b0-9f24-339f4943f31e'
                    },
                    'RetryAttempts': 0
                },
                'stopReason': 'end_turn',
                'metrics': {'latencyMs': [2453]},
                'model_provider': 'bedrock_converse',
                'model_name': 'cohere.command-r-plus-v1:0'
            },
            id='lc_run--019de9b9-221a-7c00-941a-0caa56e3ab78-0',
            tool_calls=[],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 80,
                'output_tokens': 11,
                'total_tokens': 91,
                'input_token_details': {'cache_creation': 0, 'cache_read': 0}
            }
        ),
        HumanMessage(
            content='I want to know seats available for AI cousrse',
            additional_kwargs={},
            response_metadata={},
            id='518e3f1d-e0c5-469d-9ec8-b096bfc0b2b6'
        ),
        AIMessage(
            content=[
                {'type': 'text', 'text': 'I will check the availability of the AI course and inform the user.'},
                {
                    'type': 'tool_use',
                    'name': 'check_course_availability',
                    'input': {'course_name': 'AI course'},
                    'id': 'tooluse_uNQhgi51r3Gw2o58FPdOtt'
                }
            ],
            additional_kwargs={},
            response_metadata={
                'ResponseMetadata': {
                    'RequestId': 'a4da559e-dc85-4f05-af9a-becfeae09059',
                    'HTTPStatusCode': 200,
                    'HTTPHeaders': {
                        'date': 'Sat, 02 May 2026 17:26:00 GMT',
                        'content-type': 'application/json',
                        'content-length': '401',
                        'connection': 'keep-alive',
                        'x-amzn-requestid': 'a4da559e-dc85-4f05-af9a-becfeae09059'
                    },
                    'RetryAttempts': 0
                },
                'stopReason': 'tool_use',
                'metrics': {'latencyMs': [1880]},
                'model_provider': 'bedrock_converse',
                'model_name': 'cohere.command-r-plus-v1:0'
            },
            id='lc_run--019de9b9-c102-7652-92b6-7f32c2fb57f4-0',
            tool_calls=[
                {
                    'name': 'check_course_availability',
                    'args': {'course_name': 'AI course'},
                    'id': 'tooluse_uNQhgi51r3Gw2o58FPdOtt',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 89,
                'output_tokens': 27,
                'total_tokens': 116,
                'input_token_details': {'cache_creation': 0, 'cache_read': 0}
            }
        ),
        ToolMessage(
            content='Course not found. Please ask about Data Science or Web Dev.',
            name='check_course_availability',
            id='aa2c64e8-3067-4445-89be-1dbbdb4284b7',
            tool_call_id='tooluse_uNQhgi51r3Gw2o58FPdOtt'
        ),
        AIMessage(
            content="Apologies, I can't find an AI course. Wou